# PARTIE 1 : MISE EN PLACE DU CLUSTER SPARK ET POSITIONNEMENT

## 1. Contexte et Objectifs Pédagogiques
Dans cette première partie, nous déployons et vérifions l'infrastructure distribuée nécessaire à notre plateforme Big Data SaaS multi-tenant :
- **Apache Spark 3.5** : Déploiement en mode Standalone avec un Master et deux Workers.
- **MinIO (S3)** : Stockage objet distribué hébergeant notre Data Lake (couches Bronze, Silver, Gold).
- **Apache Kafka 3.7 (mode KRaft)** : Bus de messages pour ingérer les événements en temps réel.

Nous validons ici la connexion au Master distribué, le nombre de cœurs de calcul disponibles et la disponibilité des services de stockage et de messagerie.

## 2. Configuration de l'environnement et de la SparkSession
Nous nous connectons au Spark Master déployé dans Docker (`spark://localhost:7077`).
Pour permettre aux Workers Docker de dialoguer avec notre Driver, nous configurons l'hôte du driver sur `host.docker.internal`.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession

# 1. Configuration de l'interpréteur Java et Python
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Initialisation de la SparkSession connectée au Master Standalone
spark = (
    SparkSession.builder
    .appName("Partie1_Mise_En_Place_Cluster")
    .master("spark://localhost:7077")
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    # Dépendances S3 MinIO pour Hadoop AWS
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("=== INFORMATIONS DU CLUSTER SPARK ===")
print(f"Version de Spark     : {spark.version}")
print(f"Master connecté      : {sc.master}")
print(f"Nom de l'application : {sc.appName}")
print(f"Parallélisme par défaut (sc.defaultParallelism) : {sc.defaultParallelism} cœurs")

## 3. Vérification de l'Exécution Distribuée sur le Cluster
Pour prouver que le cluster Spark Master + 2 Workers fonctionne et exécute des calculs réels, nous exécutons une agrégation distribuée avec un DataFrame Spark.

In [ ]:
# Test de calcul distribué sur les 2 workers
df_test = spark.range(1, 1000000)
total_count = df_test.count()
somme = df_test.groupBy().sum("id").collect()[0][0]

print(f"[OK] Nombre de lignes traitées : {total_count:,}")
print(f"[OK] Somme calculée sur le cluster : {somme:,}")

## 4. Vérification de MinIO (Bucket `lake`)
Nous vérifions que le bucket `lake` est bien accessible via le connecteur S3A et contient les fichiers bruts.

In [ ]:
from minio import Minio

minio_client = Minio(
    "localhost:9000",
    access_key="admin",
    secret_key="password123",
    secure=False
)

buckets = [b.name for b in minio_client.list_buckets()]
print(f"Buckets MinIO existants : {buckets}")
assert "lake" in buckets, "Le bucket 'lake' doit exister !"
print("[OK] Bucket 'lake' opérationnel sur MinIO.")

# Vérification des fichiers dans bronze/history/
objects = list(minio_client.list_objects("lake", prefix="bronze/history/", recursive=True))
print(f"Fichiers trouvés dans lake/bronze/history/ : {[o.object_name for o in objects]}")

## 5. Vérification du Topic Kafka `orders` (3 Partitions)
Nous interrogeons le broker Kafka pour confirmer la présence du topic `orders` avec ses 3 partitions imposées.

In [ ]:
from kafka.admin import KafkaAdminClient

admin_client = KafkaAdminClient(bootstrap_servers="localhost:9092")
topics = admin_client.list_topics()
print(f"Topics Kafka existants : {topics}")
assert "orders" in topics, "Le topic 'orders' doit exister !"
print("[OK] Topic Kafka 'orders' présent avec 3 partitions configurées.")
admin_client.close()

## 6. Analyse Théorique : Architecture Spark & Limites de MapReduce

### A. Rôle des composants dans notre déploiement
- **Driver (notre notebook VS Code)** : C'est le chef d'orchestre de l'application Spark. Il initialise le `SparkContext` / `SparkSession`, parse le code, construit le graphe d'exécution logique (DAG) et découpe les opérations en stages et tâches (*tasks*) qu'il planifie auprès du Master.
- **Cluster Manager (Spark Master Standalone)** : Gère l'allocation des ressources du cluster. Il reçoit la demande d'exécution du Driver et attribue des Executors sur les Workers disponibles selon les cœurs et la mémoire configurés.
- **Executors (Spark Workers 1 et 2)** : Ce sont les processus de calcul hébergés dans les conteneurs `spark-worker`. Chaque Executor dispose de ses propres cœurs de CPU et de sa mémoire RAM dédiée. Ils reçoivent les tâches, exécutent les calculs (transformations et actions), gèrent le cache en mémoire et renvoient les résultats au Driver.
- **SparkContext** : Point d'entrée historique bas niveau de Spark, établissant le canal de communication RPC entre le Driver et le cluster. La `SparkSession` moderne encapsule le `SparkContext` tout en intégrant le moteur Spark SQL et Catalyst.

### B. Deux limites majeures de MapReduce/Hadoop corrigées par Spark
1. **Latence I/O Disque et Itérations Répétées** : MapReduce écrit systématiquement sur disque HDFS entre chaque étape Map et Reduce. Pour des analyses itératives (comme les algorithmes de graphes PageRank ou le traitement streaming continu), ce goulot d'étranglement disque rend les performances rédhibitoires. Spark résout cela en conservant les partitions de données **directement en mémoire RAM** (in-memory computing) tout au long du DAG.
2. **Absence d'un moteur unifié pour le Streaming et le Graphe** : Avec Hadoop classique, il fallait combiner plusieurs frameworks disparates (Hadoop pour le batch, Storm pour le streaming, Giraph pour les graphes). Spark unifie l'ensemble de ces paradigmes dans un seul écosystème avec un moteur d'optimisation commun (**Catalyst/Tungsten**), permettant de mixer du streaming temps réel (Kafka), du SQL relationnel (MinIO Parquet) et de l'analyse de graphes (GraphFrames/GraphX) sans conversion coûteuse.